<a href="https://colab.research.google.com/github/Diggi14/project_Property2/blob/main/basemodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/Diggi14/project_Property2.git

Cloning into 'project_Property2'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 46 (delta 20), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 3.43 MiB | 7.32 MiB/s, done.
Resolving deltas: 100% (20/20), done.


In [2]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_csv('/content/project_Property2/model_trainer.csv')

In [4]:
df.head(3)

,Unnamed: 0,BEDROOM_NUM,BATHROOM_NUM,BALCONY_NUM,PRICE_PER_UNIT_AREA,FURNISH,AGE,FLOOR_NUM,SUPERBUILTUP_SQFT,PRICE,LOCALITY_WO_CITY
0,0,4.0,4.0,4.0,8766.0,Semi-furnished,5-10,high,3434.0,2.63,Sector 84
1,1,4.0,4.0,3.0,21176.0,Semi-furnished,1-5,mid,2870.0,3.60,Sector 81
2,2,3.0,3.0,3.0,13740.0,Semi-furnished,1-5,high,2802.0,3.85,Sector 112


In [5]:
df.drop('Unnamed: 0',axis=1,inplace=True)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9330 entries, 0 to 9329
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   BEDROOM_NUM          9330 non-null   float64
 1   BATHROOM_NUM         9330 non-null   float64
 2   BALCONY_NUM          9330 non-null   float64
 3   PRICE_PER_UNIT_AREA  9330 non-null   float64
 4   FURNISH              9330 non-null   object 
 5   AGE                  9330 non-null   object 
 6   FLOOR_NUM            9330 non-null   object 
 7   SUPERBUILTUP_SQFT    9330 non-null   float64
 8   PRICE                9330 non-null   float64
 9   LOCALITY_WO_CITY     9330 non-null   object 
dtypes: float64(6), object(4)
memory usage: 729.0+ KB


In [9]:
Y=df["PRICE"]
X=df.drop("PRICE",axis=1)

In [10]:
Y=np.log1p(Y)

In [6]:
from sklearn.model_selection import train_test_split,KFold,cross_val_score
from sklearn.linear_model import LinearRegression,Lasso,Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [23]:
prepreocessing=ColumnTransformer(
    transformers=[
        ('num',StandardScaler(),['BEDROOM_NUM','BATHROOM_NUM','BALCONY_NUM','PRICE_PER_UNIT_AREA','SUPERBUILTUP_SQFT']),
        ('cat',OneHotEncoder(handle_unknown='ignore'),['FURNISH','AGE','FLOOR_NUM','LOCALITY_WO_CITY'])
    ],remainder='passthrough')

In [24]:
pipeline=Pipeline([
    ('prepreocessing',prepreocessing),
    ('model',LinearRegression())
])

In [25]:
kfold=KFold(n_splits=5,shuffle=True,random_state=42)
score=cross_val_score(pipeline,X,Y,cv=kfold,scoring='r2')

In [26]:
score.mean(),score.std()

(np.float64(0.9562530883203421), np.float64(0.0026954422318578417))

In [29]:
x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

In [30]:
pipeline.fit(x_train,y_train)

Pipeline(steps=[('prepreocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['BEDROOM_NUM',
                                                   'BATHROOM_NUM',
                                                   'BALCONY_NUM',
                                                   'PRICE_PER_UNIT_AREA',
                                                   'SUPERBUILTUP_SQFT']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['FURNISH', 'AGE',
                                                   'FLOOR_NUM',
                                                   'LOCALITY_WO_CITY'])])),
                ('model', LinearRegression())])

In [31]:
y_pred=pipeline.predict(x_test)

In [32]:
y_pred=np.expm1(y_pred)

In [33]:
mean_absolute_error(y_test,y_pred)

1.514893830192923

In [34]:
from sklearn.svm import SVR

In [35]:
pipeline=Pipeline([
    ('prepreocessing',prepreocessing),
    ('model',SVR())
])

In [37]:
kfold=KFold(n_splits=5,shuffle=True,random_state=42)
score=cross_val_score(pipeline,X,Y,cv=kfold,scoring='r2')

In [38]:

score.mean(),score.std()

(np.float64(0.9689492675402838), np.float64(0.00241143978133081))

In [41]:
x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

In [43]:
pipeline.fit(x_train,y_train)
y_pred=pipeline.predict(x_test)
y_pred=np.expm1(y_pred)
mean_absolute_error(y_test,y_pred)

1.396251719525015